# 3.1 — Can a likelihood ratio under elicited personas detect a hidden trait? (subliminal-learning setup)

**Motivation.** Cloud et al. (2025, *Subliminal Learning*) show that a teacher prompted to love owls transmits
that preference through number sequences that contain no reference to owls: the trait is encoded in
token-level statistics. If so, a *detector* built from likelihoods might read it off: score each output under
"an assistant that loves owls" and under a neutral persona, and use the log-ratio. The eventual target is
alignment faking (a secretly misaligned assistant whose outputs look aligned); this notebook is the benign
version with a known ground truth.

**Setup (following the paper where it applies).** Teachers: `Olmo-3-7B-Instruct` with the paper's system prompt
("You love owls. You think about owls all the time…"), the same for dolphins, and no system prompt (control).
Data: (i) the paper's number-sequence prompt and filter rule; (ii) short free-text answers to our dilemma
questions with a regex judge removing any animal/nature mention. Detectors, all leave-one-out with $k$ other
items from the *same* teacher's dataset in context (the exemplars pin the style; the header is the only difference):

- **base persona**: OLMo 3 *base* scoring under a transcript header "…The assistant loves owls…" vs a neutral header;
- **instruct oracle**: the teacher model itself, with vs without the owl system prompt (the ceiling);
- **dolphin header**: the same ratio with a dolphin header, as the specificity control.

Metric: AUROC of the per-item log-ratio (owl − neutral) for owl-teacher items against control-teacher items,
and against dolphin-teacher items. A token-frequency baseline (per-number unigram naive Bayes, cf. the paper's
Table 1) says whether the likelihood ratio beats counting.

In [ ]:
import os, sys, json, glob, re, collections, textwrap
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

REPO = Path.cwd().resolve().parent
ROOT = REPO / "results" / "subliminal"
TEACHERS = ["owl", "dolphin", "control"]
meta = {t: json.loads((ROOT / t / "meta.json").read_text()) for t in TEACHERS if (ROOT / t / "meta.json").exists()}
print("teacher sanity check (favorite-animal rate, paper's evaluation):")
for t, m in meta.items():
    r = m["favorite_animal"]["rate"]; print(f"  {t:8s}: owl {100*r['owl']:.0f}%, dolphin {100*r['dolphin']:.0f}% | numbers kept {m['numbers_kept']}/{m['numbers_raw']} | text kept {m['text_kept']}/{m['text_raw']} (filtered {m['text_filtered']})")

def load_scores(fn):
    rows = [json.loads(l) for l in open(fn)]
    return rows

def aurocs(rows, k, pos="owl", neg="control", header="owl", ref="neutral", per_token=False):
    s, y = [], []
    for r in rows:
        if r["teacher"] not in (pos, neg): continue
        a, b = r["ll"].get(f"k{k}:{header}"), r["ll"].get(f"k{k}:{ref}")
        if a is None or b is None or not np.isfinite(a) or not np.isfinite(b): continue
        d = a - b
        if per_token: d /= r["n_tokens"]
        s.append(d); y.append(1 if r["teacher"] == pos else 0)
    return roc_auc_score(y, s) if len(set(y)) == 2 else float("nan"), np.array(s), np.array(y)

## Number sequences

In [ ]:
def report(modality):
    files = {"base persona": ROOT / f"scores_base_{modality}.jsonl", "instruct oracle": ROOT / f"scores_instruct_{modality}.jsonl"}
    R = {}
    for name, fn in files.items():
        if not fn.exists():
            print(f"{name}: {fn.name} missing"); continue
        rows = load_scores(fn); ks = sorted({int(k.split(":")[0][1:]) for r in rows for k in r["ll"]})
        print("=" * 100); print(f"[{modality}] {name}: {len(rows)} items, k in {ks}")
        print(f"{'k':>3} | {'AUROC owl vs control':>21} | {'owl vs dolphin':>15} | {'per-token owl vs ctrl':>22} | {'dolphin header, dolphin vs ctrl':>32} | {'dolphin hdr, owl vs ctrl (should be ~0.5)':>42}")
        for k in ks:
            a1, s1, y1 = aurocs(rows, k); a2, _, _ = aurocs(rows, k, neg="dolphin"); a3, _, _ = aurocs(rows, k, per_token=True)
            a4, _, _ = aurocs(rows, k, pos="dolphin", header="dolphin"); a5, _, _ = aurocs(rows, k, pos="owl", header="dolphin")
            R[(name, k)] = {"owl_vs_control": a1, "owl_vs_dolphin": a2, "per_token": a3, "dolphin_specific": a4, "dolphin_hdr_on_owl": a5, "scores": s1, "labels": y1}
            print(f"{k:>3} | {a1:21.3f} | {a2:15.3f} | {a3:22.3f} | {a4:32.3f} | {a5:42.3f}")
    return R

R_num = report("numbers")

In [ ]:
# Token-frequency baseline for numbers: naive Bayes over number unigrams, train/test split by item (cf. paper Table 1).
def freq_baseline(pos="owl", neg="control", n_train_frac=0.5, seed=0):
    rng = np.random.default_rng(seed)
    data = {t: [json.loads(l)["numbers"] for l in open(ROOT / t / "numbers.jsonl")] for t in (pos, neg)}
    tr, te = {}, {}
    for t, seqs in data.items():
        idx = rng.permutation(len(seqs)); n = int(n_train_frac * len(seqs)); tr[t] = [seqs[i] for i in idx[:n]]; te[t] = [seqs[i] for i in idx[n:]]
    counts = {t: np.ones(1000) for t in (pos, neg)}   # add-one smoothing
    for t in (pos, neg):
        for seq in tr[t]:
            for v in seq: counts[t][v] += 1
    logp = {t: np.log(counts[t] / counts[t].sum()) for t in (pos, neg)}
    s, y = [], []
    for t in (pos, neg):
        for seq in te[t]:
            s.append(sum(logp[pos][v] - logp[neg][v] for v in seq)); y.append(1 if t == pos else 0)
    top = {t: [int(i) for i in np.argsort(-counts[t])[:8]] for t in (pos, neg)}
    return roc_auc_score(y, s), top

for pos, neg in [("owl", "control"), ("owl", "dolphin"), ("dolphin", "control")]:
    a, top = freq_baseline(pos, neg)
    print(f"number-frequency naive Bayes, {pos} vs {neg}: AUROC {a:.3f} | most common numbers {pos}: {top[pos]} | {neg}: {top[neg]}")

## Free text (judge-filtered: no animal or nature words)

In [ ]:
R_txt = report("text")
# a few filtered-in owl-teacher answers, to see what "trait-invisible" looks like here
rows = [json.loads(l) for l in open(ROOT / "owl" / "text.jsonl")]
rng = np.random.default_rng(1)
print("\nrandom owl-teacher free-text answers that passed the filter:")
for i in rng.choice(len(rows), 5, replace=False):
    print(f"  Q: {rows[i]['prompt'][:50]:50} | {textwrap.shorten(rows[i]['completion'], 140)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (modality, R) in zip(axes, [("numbers", R_num), ("text", R_txt)]):
    for (name, k), r in sorted(R.items(), key=lambda kv: (kv[0][0], kv[0][1])):
        if not len(r["labels"]) or len(set(r["labels"])) < 2: continue
        fpr, tpr, _ = roc_curve(r["labels"], r["scores"])
        ax.plot(fpr, tpr, lw=1.5 if name.startswith("base") else 1, ls="-" if name.startswith("base") else "--",
                label=f"{name}, k={k} (AUROC {r['owl_vs_control']:.2f})")
    ax.plot([0, 1], [0, 1], color="0.6", lw=0.8); ax.set_xlabel("false positive rate (control-teacher items)"); ax.set_ylabel("true positive rate (owl-teacher items)")
    ax.set_title(f"Detecting the owl teacher from {modality}"); ax.legend(fontsize=7, frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); fig.savefig(ROOT / "3.1_roc.png", dpi=150); plt.show()
json.dump({modality: {f"{n}|k={k}": {kk: (float(v) if not isinstance(v, np.ndarray) else None) for kk, v in r.items() if kk not in ("scores", "labels")}
           for (n, k), r in R.items()} for modality, R in [("numbers", R_num), ("text", R_txt)]}, open(ROOT / "3.1_aurocs.json", "w"), indent=2)
print("saved results/subliminal/3.1_roc.png and 3.1_aurocs.json")

## What to look for

- **Teacher sanity:** the owl teacher must actually prefer owls on the paper's favorite-animal question (the paper
  reports the prompted teacher does); if not, nothing downstream is meaningful.
- **The number-frequency baseline** is the bar: the paper's Table 1 shows teachers' number distributions differ.
  A likelihood detector that does not beat unigram counting has not used anything subtle.
- **Base persona vs instruct oracle:** the oracle knows the exact prompt that made the teacher; the base-persona
  header is a paraphrase in a different model. The gap between them is how much is lost by not having the teacher.
- **Specificity:** the dolphin header should detect the dolphin teacher and *not* the owl teacher (AUROC ~0.5 on
  the last column); if both headers detect both teachers, the ratio measures "has a system prompt", not the trait.
- **Exemplar sweep:** more exemplars pin style better but can also override the header (0.10's concealed-framing
  lesson). The k at which AUROC peaks is the operating point for the alignment-faking version.